In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [10]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_query_heads: int, num_kv_heads: int):
        super().__init__()

        assert (num_query_heads % num_kv_heads) == 0, "Query Heads should be multiple of KV Heads"

        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads

        self.head_dim = d_model // self.num_query_heads

        self.num_queries_per_kv = self.num_query_heads // self.num_kv_heads

        self.q_proj = nn.Linear(d_model, self.num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.num_kv_heads * self.head_dim, bias=False)

        self.o_proj = nn.Linear(self.num_query_heads * self.head_dim, d_model)
    
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        """
        x shape: (batch_size, seq_len, d_model)
        mask shape: (batch_size, 1, seq_len, seq_len)
        """

        batch_size, seq_len, _ = x.shape

        # Q: (B, T, num_query_heads * head_dim)
        # K, V: (B, T, num_kv_heads * head_dim)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # q shape = batch_size, self.num_query_heads, seq_len, self.head_dim
        q = q.view(batch_size, seq_len, self.num_query_heads, self.head_dim).transpose(1, 2)

        # k and v shape = batch_size, self.num_kv_heads, seq_len, self.head_dim
        k = k.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # 3. The GQA Magic: Repeat K and V to match the number of Query heads
        # This expands (B, num_kv_heads, T, head_dim) -> (B, num_query_heads, T, head_dim)
        # It literally duplicates the KV heads so each query in a group gets a copy
        print(f'Matrix K before: {k.shape}')
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        print(f'Matrix K after: {k.shape}')
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)

        # 4. Standard Scaled Dot-Product Attention
        # We can use PyTorch's highly optimized functional attention here.
        # It handles the scaling (1 / sqrt(head_dim)) and masking internally.
        is_causal = mask is None # If no explicit mask, assume causal (autoregressive)

        # out shape = batch_size, self.num_query_heads, seq_len, self.head_dim
        out = F.scaled_dot_product_attention(
            query=q,
            key=k,
            value=v,
            attn_mask=mask,
            is_causal=is_causal
        )

        # 5. Reshape back to continuous sequence
        # (B, num_query_heads, T, head_dim) -> (B, T, num_query_heads, head_dim)
        out = out.transpose(1, 2).contiguous()
        
        # Flatten the heads back into d_model
        # (B, T, num_query_heads * head_dim)
        out = out.view(batch_size, seq_len, self.d_model)

        return self.o_proj(out)

In [11]:
if __name__ == "__main__":
    batch_size = 2
    seq_len = 10
    d_model = 512
    num_query_heads = 8
    num_kv_heads = 2 # 8/2 = 4 queries share 1 KV head (GQA)
    
    # Create fake input data
    x = torch.randn(batch_size, seq_len, d_model)
    
    # Initialize GQA
    gqa = GroupedQueryAttention(d_model, num_query_heads, num_kv_heads)
    
    # Run forward pass
    output = gqa(x)
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}") 
    # Output should perfectly match input shape: [2, 10, 512]

Matrix K before: torch.Size([2, 2, 10, 64])
Matrix K after: torch.Size([2, 8, 10, 64])
Input shape: torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
